# 1.    ANNOTATION STRUCTURELLE LAURUS AZORICA

#### 1.0 Connexion genotoul

In [ ]:
# --- Connexion genotoul SSH ---
ssh tbertrand@genobioinfo2.toulouse.inrae.fr

## 1.1  Annotation structurelle avec BRAKER 3

### Construction prot database

Telechargement data proteines OrthoDB12 Viridiplantae \
On filtre pour garder uniquement les Embryophytes \
voir le github > https://github.com/tomasbruna/orthodb-clades/tree/master

In [ ]:
cd /Users/cibio/Desktop/Laurus/Annotation/Proteines/OrthoDB

# --- Télécharger les fichiers OrthoDB v12 ---
wget https://bioinf.uni-greifswald.de/bioinf/partitioned_odb12/Viridiplantae.fa.gz
wget https://data.orthodb.org/current/download/odb_data_dump/odb12v2_species.tab.gz
wget https://data.orthodb.org/current/download/odb_data_dump/odb12v2_levels.tab.gz
wget https://data.orthodb.org/current/download/odb_data_dump/odb12v2_level2species.tab.gz

# --- Décompresser ---
gunzip -k Viridiplantae.fa.gz
gunzip -k odb12v2_species.tab.gz
gunzip -k odb12v2_levels.tab.gz
gunzip -k odb12v2_level2species.tab.gz

# --- Télécharger selectClade.py (outil officiel) ---
wget -O selectClade.py https://raw.githubusercontent.com/tomasbruna/orthodb-clades/master/selectClade.py
chmod +x selectClade.py

# --- Extraire uniquement les Embryophytes ---
python3 selectClade.py \
    Viridiplantae.fa \
    odb12v2_levels.tab \
    odb12v2_level2species.tab \
    Embryophyta > Embryophyta.fa

# --- Import genotoul --- 

scp /Users/cibio/Desktop/Laurus/Annotation/Proteines/OrthoDB/Embryophyta.fa \
tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/input/


###  Telechargement ref genome masked + RNAseq data Azorica

In [ ]:
# --- Copie du ref genome (nb: pas les droits sur le genome de reference) ---
scp -r tristan@162.38.181.237:/data2/fabien/L_azorica/reference_genome /home/tbertrand/work/L_azorica/

# --- Copie des reads RNAseq ---
scp -r tristan@162.38.181.237:/data2/fabien/L_azorica/RNA_data /home/tbertrand/work/L_azorica/

###  Script Braker 3 v1.2

In [ ]:
#!/bin/bash
#SBATCH --job-name=Annotation_braker3
#SBATCH -p unlimitq
#SBATCH --time=7-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=12

#############################
# 1. MODULES
#############################
module purge
module load devel/python/Python-3.6.3
module load compilers/gcc/12.2.0
module load bioinfo/BRAKER/3.0.7

#############################
# 2. FILES 
#############################

GENOME=/home/tbertrand/work/L_azorica/reference_genome/GCA_977007225.1_dmLauAzor1.pri_genomic.fna.masked
PROTEINS=/home/tbertrand/work/annotation/BRAKER/input/Embryophyta.fa
RNA_DIR=/home/tbertrand/work/L_azorica/RNA_data
RNA_IDS="LaF1,LaF2,LaF3,LaM1,LaM3,LaM4"
WORKDIR=/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir

#############################
# 3. RUN BRAKER
#############################

braker.pl \
    --genome=${GENOME} \
    --prot_seq=${PROTEINS} \
    --rnaseq_sets_dirs=${RNA_DIR} \
    --rnaseq_sets_ids=${RNA_IDS} \
    --threads=${SLURM_CPUS_PER_TASK} \
    --workingdir=${WORKDIR}


Probleme de memoire a cause de de l etape d aligement... creation de fichier SAM intermediuare de 40Go donc saturation memoire au bout de 16 individus.

Option 1: Reduire le nombre d'echamtillon RNAseq (6 par ex)

Option 2: Alignement a part puis RUN a partir des BAM merge. NB: https://github.com/Gaius-Augustus/BRAKER/issues/778

## 1.2 Results Annotation structurelle avec BRAKER 3

#### Conversion fichier 

Il faut changer les noms des contigs dans le gtf car il ne correpondes pas a ceux dans le fasta. 

Contig fasta ex: >OZ345840.1 Laurus azorica genome assembly, chromosome: 1 \
Contig fasta ex: OZ345840.1_Laurus_azorica_genome_assembly,_chromosome:_1 \

On modifie donc le gtf avec le script awk ci dessous pour garder uniquement OZ345840.1

In [ ]:
# 1. Fix contig name
awk '                                                                                               
BEGIN { OFS="\t" }
# Conserver les lignes de commentaires
/^#/ { print; next }
{
    # Extraire le contig d origine = tout jusqu au premier espace
    split($1, a, "_")
    $1 = a[1]
    print
}' \
/Users/cibio/Desktop/Laurus/Annotation/run_1_2_workdir/braker.gtf \
> /Users/cibio/Desktop/Laurus/Annotation/run_1_2_workdir/braker.fixed_ids.gtf

Instalation AGAT + Conversion Braker.gtf en braker.gff3

In [ ]:
conda create -n agat_env python=3.10
conda activate agat_env
conda install -c bioconda -c conda-forge agat augustus

agat_convert_sp_gxf2gxf.pl -g braker.fixed_ids.gtf -o braker.agat.gff3

### Alignement

In [ ]:
#Copie des resultats en local

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir /Users/cibio/Desktop/Laurus/Annotation/
scp tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/Braker_L_azorica_v1_2.sh /Users/cibio/Desktop/Laurus/Annotation/

#Voir (run_1_2_workdir/errors/hisat2.LaF1.stderr)

LaF1 : 86.40% overall alignment rate \
LaF2 : 89.01% overall alignment rate \
LaF3 : 89.30% overall alignment rate 

LaM1 : 82.51% overall alignment rate \
LaM3 : 87.75% overall alignment rate \
LaM4 : 89.67% overall alignment rate 

In [ ]:
grep -c ">" braker.aa > nb_of_prot_seq.txt

# 34295 prot


### Stats gFACs

Installation gFACs (https://gitlab.com/PlantGenomicsLab/gFACs)

In [ ]:
# Creatye conda environement
CONDA_SUBDIR=osx-64 conda create -n gfacs -c conda-forge -c bioconda perl-bioperl
conda activate gfacs

In [ ]:
#RUN stats gFACs 
cd /Users/cibio/Desktop/Laurus/Annotation/Results 

/Users/cibio/Desktop/Laurus/Annotation/gFACs-master/gFACs.pl -f braker_2.0_gtf --statistics --fasta /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fna --splice-table --nt-content \
	--get-fasta --get-protein-fasta --create-gff3 --distributions exons_lengths intron_lengths CDS_lengths gene_lengths \
	exon_position exon_position_data intron_position intron_position_data -O gFACs /Users/cibio/Desktop/Laurus/Annotation/run_1_2_workdir/braker.gtf

STATS:
Number of genes:	34348
Number of monoexonic genes:	8073
Number of multiexonic genes:	26222

Number of positive strand genes:	17398
Monoexonic:	4205
Multiexonic:	13140

Number of negative strand genes:	16950
Monoexonic:	3868
Multiexonic:	13082

Average overall gene size:	10108.776
Median overall gene size:	4004
Average overall CDS size:	1280.457
Median overall CDS size:	1041
Average overall exon size:	237.456
Median overall exon size:	129

Average size of monoexonic genes:	955.862
Median size of monoexonic genes:	681
Largest monoexonic gene:	13047
Smallest monoexonic gene:	33

Average size of multiexonic genes:	12946.713
Median size of multiexonic genes:	6376
Largest multiexonic gene:	688020
Smallest multiexonic gene:	77

Average size of multiexonic CDS:	1382.978
Median size of multiexonic CDS:	1128
Largest multiexonic CDS:	16407
Smallest multiexonic CDS:	9

Average size of multiexonic exons:	204.716
Median size of multiexonic exons:	123
Average size of multiexonic introns:	2005.130
Median size of multiexonic introns:	501

Average number of exons per multiexonic gene:	6.756
Median number of exons per multiexonic gene:	5
Largest multiexonic exon:	13130
Smallest multiexonic exon:	1
Most exons in one gene:	79

Average number of introns per multiexonic gene:	5.756
Median number of introns per multiexonic gene:	4
Largest intron:	461357
Smallest intron:	20

The following columns do not involve codons:
Number of complete models:	30371
Number of 5' only incomplete models:	2363
Number of 3' only incomplete models:	1472
Number of 5' and 3' incomplete models:	89

### Stats agat

In [ ]:
#!/bin/bash
#SBATCH --job-name=agat_stat
#SBATCH -p workq
#SBATCH --time=10:00:00
#SBATCH --mem=10G
#SBATCH --cpus-per-task=4
#SBATCH --mail-type=END,FAIL

module load devel/Miniconda/Miniconda3
module load bioinfo/AGAT/1.4.1

agat_sp_statistics.pl \
  --gff /home/tbertrand/work/stat_annoatation/braker.fixed_ids.gff3 \
  -o /home/tbertrand/work/stat_annoatation/braker_fixed_ids_stats.txt 


#### results AGAT

In [ ]:
---------------------------------- transcript ----------------------------------
Number of gene                                              27900
Number of transcript                                        31505
Number gene overlapping                                     0
Number of single exon gene                                  27900
mean transcripts per gene                                   1.1
Total gene length (bp)                                      248728282
Total transcript length (bp)                                285967017
mean gene length (bp)                                       8915
mean transcript length (bp)                                 9077
median gene length (bp)                                     3323
median transcript length (bp)                               3473
90 percentile gene length (bp)                              20138
90 percentile transcript length (bp)                        20665
Longest gene (bp)                                           688020
Longest transcript (bp)                                     688020
Shortest gene (bp)                                          33
Shortest transcript (bp)                                    33


###  BUSCO BRAKER 3

#### FILE PROCCESIBG BUSCO

Pour lancer BUSCO en mode transcriptome, il faut d'abord faire en sorte de ne sélectionner qu'un isoforme (= transcrit) par gène. On utilise donc AGAT pour ne garder que les plus isoformes aux plus longs CDS.

In [ ]:
mkdir without_isoforms

agat_sp_keep_longest_isoform.pl \
  -gff /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/braker.fixed_ids.gtf \
  -o /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/agat_without_isoforms.gtf

Il faut ensuite reformater le fichier produit par AGAT pour qu'il corresponde à un format Augustus, ce qui nous permettra de lancer ensuite le script d'Augustus permettat d'obtenir les séquences codantes et protéiques d'une annotation.

In [ ]:
cd without_isoforms

python3 /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/reformat_agat_gtf.py \
    /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/agat_without_isoforms.gtf \
    /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/without_isoforms.gtf

getAnnoFastaFromJoingenes.py -g /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fna.masked \
	-o /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms -f /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/without_isoforms.gtf

Le script produit deux fichiers : without_isoforms.aa et without_isoforms.codingseq. \
On peut enfin lancer BUSCO sur le fichier .codingseq obtenu via genotoul

In [ ]:
scp /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/without_isoforms/without_isoforms.codingseq tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/BUSCO
scp /Users/cibio/Desktop/Laurus/Annotation/Results/BUSCO/script_busco_annotation_1.sh tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/BUSCO

#### SCRIPT BUSCO BRAKER

In [ ]:
#!/bin/bash
#SBATCH --job-name=BUSCO_Annotation_braker3
#SBATCH -p unlimitq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=8

#############################
# 1. MODULES
#############################
module purge
module load devel/Miniforge/Miniforge3
module load bioinfo/BUSCO/5.8.3
``

#############################
# 2. FILES 
#############################

CODINGSEQ=/home/tbertrand/work/annotation/BUSCO/without_isoforms.codingseq
WORKDIR=/home/tbertrand/work/annotation/BRAKER/BUSCO/Run_busco_1

#############################
# 3. RUN BUSCO
#############################

busco -m transcriptome \
    -i ${CODINGSEQ} \
    -o busco_results \
    --out_path ${WORKDIR} \
    -l embryophyta_odb10 \
    -c 8



In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/BUSCO /Users/cibio/Desktop/Laurus/Annotation/BRAKER/Results/BUSCO

#### RESULTS BUSCO

BUSCO version is: 5.8.3 
The lineage dataset is: embryophyta_odb10 (Creation date: 2024-01-08, number of genomes: 50, number of BUSCOs: 1614)
Summarized benchmarking in BUSCO notation for file /home/tbertrand/work/annotation/BUSCO/without_isoforms.codingseq
BUSCO was run in mode: euk_tran

	***** Results: *****

	C:97.3%[S:92.9%,D:4.4%],F:0.7%,M:2.0%,n:1614	   
	1570	Complete BUSCOs (C)			   
	1499	Complete and single-copy BUSCOs (S)	   
	71	Complete and duplicated BUSCOs (D)	   
	11	Fragmented BUSCOs (F)			   
	33	Missing BUSCOs (M)			   
	1614	Total BUSCO groups searched		   

Dependencies and versions:
	hmmsearch: 3.4
	metaeuk: 7.bba0d80
	python: sys.version_info(major=3, minor=10, micro=17, releaselevel='final', serial=0)
	busco: 5.8.3


## 1.3  Annotation avec HELIXER

#### Instalation HELIXER

Telechargement du model land plant en local

In [ ]:
#TELECHARGEMENT MODEL 

module load devel/Miniforge/Miniforge3
module load bioinfo/Helixer/0.3.6

mkdir /home/tbertrand/work/annotation/HELIXER/models

# Téléchargement des modèles via le script fourni par Helixer
fetch_helixer_models.py -l land_plant -c /home/tbertrand/work/annotation/HELIXER/models

#### Script HELIXER 1

In [ ]:
#!/bin/bash
#SBATCH --job-name=Annotation_Helixer

#SBATCH --partition=gpuq
#SBATCH --gres=gpu:1 

#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=12

#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL


#############################
# 1. MODULES
#############################

module purge
module load devel/Miniforge/Miniforge3
module load bioinfo/Helixer/0.3.6


#############################
# 2. FILES 
#############################

MODEL_DIR="/home/tbertrand/work/annotation/HELIXER/models"
GENOME=/home/tbertrand/work/L_azorica/reference_genome/GCA_977007225.1_dmLauAzor1.pri_genomic.fna
OUTDIR=/home/tbertrand/work/annotation/HELIXER/run_HELIXER_1

#############################
# 3. RUN HELIXER
#############################

Helixer.py --lineage land_plant \
    --downloaded-model-path "${MODEL_DIR}" \
    --fasta-path ${GENOME}  \
    --species Laurus_azorica \
    --gff-output-path ${OUTDIR}/Helixer.gff3 

In [ ]:
scp /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fna.gz tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_azorica/reference_genome 

## 1.4  RESULTS HELIXER

### Stats gFACs HELIXER

In [ ]:
#Copie des resultats en local

scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/HELIXER/ /Users/cibio/Desktop/Laurus/Annotation/HELIXER

In [ ]:
conda activate gfacs

/Users/cibio/Desktop/Laurus/Annotation/gFACs-master/gFACs.pl \
    -f gffread_0.9.12_gff3 \
    --statistics \
    --fasta /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fna.gz \
    --splice-table \
    --nt-content  \
    --create-gff3 \
    --distributions exons_lengths intron_lengths CDS_lengths gene_lengths \
                   exon_position exon_position_data intron_position intron_position_data \
    -O /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Results/gFACs \
    /Users/cibio/Desktop/Laurus/Annotation/HELIXER/HELIXER/run_HELIXER_1/Helixer.gff3

STATS:
Number of genes:	62278
Number of monoexonic genes:	7436
Number of multiexonic genes:	32269

Number of positive strand genes:	45147
Monoexonic:	5353
Multiexonic:	17221

Number of negative strand genes:	17131
Monoexonic:	2083
Multiexonic:	15048

Average overall gene size:	6334.735
Median overall gene size:	4907
Average overall CDS size:	973.872
Median overall CDS size:	1316
Average overall exon size:	268.139
Median overall exon size:	123

Average size of monoexonic genes:	982.850
Median size of monoexonic genes:	658
Largest monoexonic gene:	14033
Smallest monoexonic gene:	-3490

Average size of multiexonic genes:	11858.903
Median size of multiexonic genes:	4628
Largest multiexonic gene:	174465
Smallest multiexonic gene:	-41666

Average size of multiexonic CDS:	1651.919
Median size of multiexonic CDS:	1447
Largest multiexonic CDS:	40345
Smallest multiexonic CDS:	104

Average size of multiexonic exons:	243.676
Median size of multiexonic exons:	118
Average size of multiexonic introns:	1076.566
Median size of multiexonic introns:	278

Average number of exons per multiexonic gene:	6.779
Median number of exons per multiexonic gene:	5
Largest multiexonic exon:	21438
Smallest multiexonic exon:	1
Most exons in one gene:	87

Average number of introns per multiexonic gene:	5.779
Median number of introns per multiexonic gene:	4
Largest intron:	48463
Smallest intron:	30

The following columns do not involve codons:
Number of complete models:	0
Number of 5' only incomplete models:	0
Number of 3' only incomplete models:	0
Number of 5' and 3' incomplete models:	39704

In [ ]:
#!/bin/bash
#SBATCH --job-name=agat_stat
#SBATCH -p workq
#SBATCH --time=10:00:00
#SBATCH --mem=2G
#SBATCH --cpus-per-task=1
#SBATCH --mail-type=END,FAIL

module load devel/Miniconda/Miniconda3
module load bioinfo/AGAT/1.4.1

agat_sp_statistics.pl \
  --gff /home/tbertrand/work/stat_annoatation/braker.fixed_ids.gff3 \
  -o /home/tbertrand/work/stat_annoatation/braker_fixed_ids_stats.txt 

scp -r /home/tbertrand/work/stat_annoatation

### BUSCO HELIXER

Pas besoin de filtrer les isoformes pour HELIXER car il ya seuelement 1 transcript par gene. 

Il faut juste extraire les CDS dans un fichier fasta pour l'input de BUSCO.


In [ ]:
gffread /Users/cibio/Desktop/Laurus/Annotation/HELIXER/HELIXER/run_HELIXER_1/Helixer.gff3 \
  -g /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fasta \
  -y /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Results/BUSCO/helixer.protein.fa

gffread /Users/cibio/Desktop/Laurus/Annotation/HELIXER/HELIXER/run_HELIXER_1/Helixer.gff3 \
  -g /Users/cibio/Desktop/Laurus/genome/L_azorica/GCA_977007225.1_dmLauAzor1.pri_genomic.fasta \
  -x /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Results/BUSCO/helixer.cds.fa


In [ ]:
scp /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Results/BUSCO/helixer.cds.fa tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/HELIXER/BUSCO
scp /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Results/BUSCO/helixer.protein.fa tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/HELIXER/BUSCO

#### Script BUSCO HELIXER GENOTOUL

In [ ]:
#!/bin/bash
#SBATCH --job-name=BUSCO_Annotation_HELIXER_CDS
#SBATCH -p workq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=8

#############################
# 1. MODULES
#############################
module purge
module load devel/Miniforge/Miniforge3
module load bioinfo/BUSCO/5.8.3
``

#############################
# 2. FILES 
#############################

CODINGSEQ=/home/tbertrand/work/annotation/HELIXER/BUSCO/helixer.cds.fa
WORKDIR=/home/tbertrand/work/annotation/HELIXER/BUSCO/Run_BUSCO_PROT_1

#############################
# 3. RUN BUSCO
#############################

busco -m transcriptome \
    -i ${CODINGSEQ} \
    -o busco_results \
    --out_path ${WORKDIR} \
    -l embryophyta_odb10 \
    -c 8

#### RESULATS BUSCO HELIXER

BUSCO version is: 5.8.3 \
The lineage dataset is: embryophyta_odb10 (Creation date: 2024-01-08, number of genomes: 50, number of BUSCOs: 1614) \
Summarized benchmarking in BUSCO notation for file /home/tbertrand/work/annotation/HELIXER/BUSCO/helixer.cds.fa \
BUSCO was run in mode: euk_tran \

	***** Results: *****

	C:84.6%[S:80.0%,D:4.6%],F:10.5%,M:4.8%,n:1614	   
	1366	Complete BUSCOs (C)			   
	1291	Complete and single-copy BUSCOs (S)	   
	75	Complete and duplicated BUSCOs (D)	   
	170	Fragmented BUSCOs (F)			   
	78	Missing BUSCOs (M)			   
	1614	Total BUSCO groups searched		   

Dependencies and versions: \
	hmmsearch: 3.4 \
	metaeuk: 7.bba0d80 \
	python: sys.version_info(major=3, minor=10, micro=17, releaselevel='final', serial=0) \
	busco: 5.8.3 \

In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/HELIXER/BUSCO /Users/cibio/Desktop/Laurus/Annotation/HELIXER/Resultsscp

HELIXER prédit un nombre de gènes nettement plus élevé que BRAKER et la qualité globale de son annotation est inférieure selon les scores BUSCO.
Le taux de complétude BUSCO est plus bas pour HELIXER, ce qui indique qu’il manque davantage de gènes attendus.
À l’inverse, BRAKER produit moins de gènes, mais ceux‑ci semblent plus complets, plus cohérents et mieux alignés avec les gènes conservés évalués par BUSCO.

## 1.5 Comparaison BRAKER | HELIXER

Conversion du gtf braker avec les bons ID en gff

In [ ]:
cd /Users/cibio/Desktop/Laurus/Annotation/Compare

agat_convert_sp_gxf2gxf.pl -g braker.fixed_ids.gtf -o braker.agat.gff3

Comparaison des deux annotation en alternant les references

In [ ]:
gffcompare \
  -r HELIXER.gff3 \
  -o HELIXER_vs_BRAKER \
  braker.agat.gff3

gffcompare \
  -r braker.agat.gff3 \
  -o BRAKER_VS_HELIXER \
  HELIXER.gff3

## 1.6 Comparaison des BUSCO BRAKER et HELIXER

L’objectif est d’identifier les gènes BUSCO qui sont détectés par HELIXER mais absents dans l’annotation BRAKER.
Ces gènes manquants représentent des gènes hautement conservés dont la présence est attendue dans tout génome complet et correctement annoté.
Pour améliorer la complétude de l’annotation finale, il est donc pertinent d’intégrer dans BRAKER les gènes BUSCO que seul HELIXER parvient à prédire.
Cela permet d’obtenir une annotation plus exhaustive, tout en conservant la qualité globale supérieure fournie par BRAKER.

### R script comparaison des BUSCO

In [ ]:
############################################
## COMPARAISON DES FULL_TABLE BUSCO EN R  ##
############################################

library(dplyr)
library(readr)

braker_BUSCO_file <- "~/Desktop/Laurus/Annotation/BRAKER/Results/BUSCO/Run_busco_1/busco_results/run_embryophyta_odb10/full_table.tsv"
helixer_BUSCO_file <- "~/Desktop/Laurus/Annotation/HELIXER/Results/BUSCO/Run_BUSCO_CDS_1/busco_results/run_embryophyta_odb10/full_table.tsv"

braker_BUSCO_all <- read_tsv(braker_BUSCO_file, comment = "#",
                         col_names = c("Busco_id","Status","Sequence","Score","Lenght","OrthoDB_url","Description"),
                         col_types = cols(.default="c"))

helixer_BUSCO_all <- read_tsv(helixer_BUSCO_file, comment="#",
                          col_names = c("Busco_id","Status","Sequence","Score","Lenght","OrthoDB_url","Description"),
                          col_types = cols(.default="c"))

# On garde seulement busco_id + statut
braker_BUSCO <- braker_BUSCO_all %>% select(Busco_id, Status) %>% mutate(tool="BRAKER")
helixer_BUSCO <- helixer_BUSCO_all %>% select(Busco_id, Status) %>% mutate(tool="HELIXER")

# On garde seulemement 1 ligne par BUSCO duplique

braker_BUSCO <- braker_BUSCO %>%
  arrange(Status) %>%
  distinct(Busco_id, .keep_all = TRUE)

helixer_BUSCO <- helixer_BUSCO %>%
  arrange(Status) %>%
  distinct(Busco_id, .keep_all = TRUE)

# Jointure 
compare_BUSCO <- full_join(
  braker_BUSCO %>% select(Busco_id, Status_braker = Status),
  helixer_BUSCO %>% select(Busco_id, Status_helixer = Status),
  by = "Busco_id"
)

# BUSCO IDs Missing dans BRAKER mais Complete dans HELIXER
Missing_BUSCO <- compare_BUSCO %>%
  filter(Status_braker == "Missing", Status_helixer == "Complete") %>%
  arrange(Busco_id)

Missing_BUSCO_helixer_sequence_ids <- helixer_BUSCO_all %>%
  filter(Busco_id %in% Missing_BUSCO$Busco_id) %>%
  pull(Sequence) %>%
  unique()

# Enlever la partie :XXX-YYY et le .1 .2 des transcripts
Missing_BUSCO_helixer_sequence_ids <- sub(":.*$", "", Missing_BUSCO_helixer_sequence_ids)
print(Missing_BUSCO_helixer_sequence_ids)

Missing_BUSCO_helixer_sequence_ids <- sub("\\.[0-9]+$", "", Missing_BUSCO_helixer_sequence_ids)
print(Missing_BUSCO_helixer_sequence_ids)


writeLines(Missing_BUSCO_helixer_sequence_ids,
           "/Users/cibio/Desktop/Laurus/Annotation/Compare/Missing_BUSCO_helixer_sequence_ids")


#### Script : extract_ids_from_gff.sh

PAS NECESSAIRE ON GARDE SEULEMENT le RUN BRAKER3

In [ ]:
#!/bin/bash

ID_FILE="$1"       # fichier avec les IDs (1 par ligne)
GFF_FILE="$2"      # gff à rechercher
OUT_FILE="$3"      # fichier de sortie

> "$OUT_FILE"       # vide le fichier output

while read -r ID; do
    grep "$ID" "$GFF_FILE" >> "$OUT_FILE"
done < "$ID_FILE"

In [ ]:
chmod +x extract_ids_from_gff.sh

./extract_ids_from_gff.sh \
  Missing_BUSCO_helixer_sequence_ids.txt \
  Helixer.gff3 \
  Missing_BUSCO_helixer.gff3

Ajout des BUSCO unique chez HLIXER (9 genes)

In [ ]:
cat braker.agat.gff3 Missing_BUSCO_helixer.gff3 > Braker_plus_BUSCO_Helixer.gff3

# 2. ANNOTATION FONCTIONELLE LAURUS AZORICA

On conserve uniquement l annotation produite par BRAKER.

## 2.1 EggNOG

Documentation : https://github.com/eggnogdb/eggnog-mapper

module load bioinfo/eggNOG-mapper/2.1.13

In [ ]:
#!/bin/bash
#SBATCH --job-name=Egg_NOG_run_1
#SBATCH -p workq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=15

#############################
# 1. MODULES
#############################

module load bioinfo/eggNOG-mapper/2.1.13

#############################
# 2. FILES 
#############################

INPUT_AA=/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir/braker.aa
WORKDIR=/home/tbertrand/work/annotation/fonctional_annotation/EggNog

#############################
# 3. RUN BUSCO
#############################

emapper.py  --sensmode more-sensitive --cpu 15 \
	-i ${INPUT_AA} \
	-o Braker_plus_BUSCO_Helixer \
    --output_dir ${WORKDIR}



In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/fonctional_annotation/EggNog /Users/cibio/Desktop/Laurus/Annotation/Annotation_fonctionelle

## 2.2  Results EggNog

Results: 32 548 proteines annotes sur 34 304  --> 94.88% de prot annotes

In [ ]:
rsync -avhn \
/Users/cibio/Desktop/Laurus/Annotation/BRAKER/run_1_2_workdir/ \
tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir

In [ ]:
scp /Users/cibio/Desktop/Laurus/Annotation/BRAKER/run_1_2_workdir/braker.fixed_ids.gff3 \
tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir/


scp /Users/cibio/Desktop/Laurus/Annotation/BRAKER/run_1_2_workdir/braker.fixed_ids.gtf \
tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/BRAKER/run_1_2_workdir/

In [ ]:
scp -r /Users/cibio/Desktop/Laurus/Annotation/Compare \
tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/annotation/

scp -r /home/tbertrand/work/annotation/ tristan@162.38.181.237:/home/tristan
